# CIP alert walkthrough: one alerting cycle

A guided check of a single cycle, from the raw PLC records to each alert it raised. Use it with the CIP owner to confirm that **the data, the thresholds and the alert logic** are right.

**Cycle:** `C02C01-PMT TANK-1`, cycle 14, 17–18 Jul 2026. Change `WALK_OBJECT` and `WALK_CYCLE` in section 2 to walk through another one.

## 1. Run the alerting pipeline
Runs every cell of `denon_cip_alerting_clean.ipynb` (same logic and validation checks) without showing its output.

In [ ]:
import contextlib
import io
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

PIPELINE = 'denon_cip_alerting_clean.ipynb'
show = display
display = lambda *args, **kwargs: None  # silence the pipeline's own tables
try:
    with contextlib.redirect_stdout(io.StringIO()):
        for cell in json.load(open(PIPELINE))['cells']:
            if cell['cell_type'] == 'code':
                exec(''.join(cell['source']))
finally:
    display = show

pipeline_checks = pd.DataFrame(validation_log)['Status'].value_counts()
print(f"{len(cycle_results):,} cycles, {len(alerts):,} alerts. Pipeline checks: "
      + ', '.join(f'{count} {status}' for status, count in pipeline_checks.items()))

## 2. The selected cycle
Chosen as a fair test case: matched to the workbook, one complete CIP run with every phase, no missing samples, and several alert types. Final Rinse passes, which gives a contrast.

In [ ]:
WALK_OBJECT = 'C02C01-PMT TANK-1'
WALK_CYCLE = 14

selected = cycle_results[cycle_results['Object'].eq(WALK_OBJECT) & cycle_results['Cycle Number'].eq(WALK_CYCLE)]
if len(selected) != 1:
    raise ValueError(f'{WALK_OBJECT} cycle {WALK_CYCLE} not found')
cycle = selected.iloc[0]
records = logs[logs['Object'].eq(WALK_OBJECT) & logs['Cycle Number'].eq(WALK_CYCLE)].copy()
cycle_alerts = alerts[alerts['Object'].eq(WALK_OBJECT) & alerts['Cycle Number'].eq(WALK_CYCLE)]
planned = process.set_index('Object').loc[cycle['Reference Object']]
by_activity = records.groupby('Activity', sort=False)
activity_order = records['Activity'].drop_duplicates().tolist()

pd.DataFrame([
    ('Matched to a workbook object', pd.notna(cycle['Reference Object'])),
    ('Cycle is in ALERT status', cycle['Status'] == 'ALERT'),
    ('One CIP run (elapsed counter never restarts)', records['Time In Seconds'].diff().dropna().ge(0).all()),
    ('No missing one-minute samples', records['Date Time'].diff().dt.total_seconds().dropna().le(90).all()),
    ('Every phase present (Pre Rinse to Final Rinse)', cycle[ACTIVITIES[:5]].notna().all()),
    ('More than one alert type', cycle_alerts['Alert Type'].nunique() > 1),
], columns=['Criterion', 'Met'])

## 3. Cycle card

In [ ]:
cycle_minutes = (cycle['Cycle End'] - cycle['Cycle Start']).total_seconds() / 60
pd.Series({
    'Line': cycle['Line'],
    'Object (log)': WALK_OBJECT,
    'Object (workbook)': cycle['Reference Object'],
    'Matched on equipment code': cycle['Object Key'],
    'Recipe': cycle['Recipes'],
    'Start': f"{cycle['Cycle Start']:%a %d %b %Y, %H:%M}",
    'End': f"{cycle['Cycle End']:%a %d %b %Y, %H:%M}",
    'Duration': f'{cycle_minutes:.0f} min',
    'Records': f"{cycle['Record Count']} (one per minute)",
    'Activities': ' → '.join(activity_order),
    'Source': f"{records['Source File'].iloc[0]}, Excel rows {records['Excel Row'].min()}–{records['Excel Row'].max()}",
    'Status': f"{cycle['Status']}: {cycle['Alert Count']} breaches",
    'Alert types': ', '.join(cycle_alerts['Alert Type']),
}, name='Value').to_frame()

## 4. Alerts raised
Each alert with the limit it was tested against (threshold ± tolerance) and how far the cycle went past it.

In [ ]:
alert_view = cycle_alerts[['Alert Type', 'Actual Value', 'Threshold', 'Unit']].reset_index(drop=True)
is_duration = alert_view['Unit'].eq('minutes')
alert_view['Limit'] = np.where(is_duration, alert_view['Threshold'] * (1 + DURATION_TOLERANCE),
                               alert_view['Threshold'] * (1 - FLOW_TOLERANCE))
alert_view['Vs Threshold'] = (alert_view['Actual Value'] / alert_view['Threshold'] - 1).map('{:+.0%}'.format)
alert_view['Rule'] = np.where(is_duration, f'time > threshold × {1 + DURATION_TOLERANCE:.2f}',
                              f'median flow < threshold × {1 - FLOW_TOLERANCE:.2f}')
alert_view.round(1)

## 5. Activity times vs thresholds
- **Notebook time** = last counter − first counter in the activity. This is what the alerts use.
- **Elapsed time** runs until the next activity starts, so it also counts the last minute.
- The **Acid** threshold covers Mid Rinse + Acid, so that sum is shown as an extra row.

In [ ]:
counters = by_activity['Time In Seconds'].agg(['first', 'last'])
activity_table = pd.DataFrame({
    'Start': by_activity['Date Time'].min().dt.strftime('%H:%M'),
    'Records': by_activity.size(),
    'Notebook (min)': (counters['last'] - counters['first']) / 60,
    'Elapsed (min)': (counters['first'].shift(-1) - counters['first']) / 60,
})
activity_table['Elapsed (min)'] = activity_table['Elapsed (min)'].fillna(activity_table['Records'])  # last activity: one record per minute
activity_table.loc['Mid Rinse + Acid'] = ['', activity_table.loc[['Mid Rinse', 'Acid'], 'Records'].sum(),
                                          activity_table.loc[['Mid Rinse', 'Acid'], 'Notebook (min)'].sum(),
                                          activity_table.loc[['Mid Rinse', 'Acid'], 'Elapsed (min)'].sum()]
threshold_for = {**{phase: cycle[f'{phase} Threshold (min)'] for phase in DURATION_THRESHOLDS},
                 'Mid Rinse + Acid': cycle['Acid Threshold (min)']}
activity_table['Threshold (min)'] = activity_table.index.map(threshold_for)
activity_table['Limit (min)'] = activity_table['Threshold (min)'] * (1 + DURATION_TOLERANCE)
activity_table['Breach'] = np.where(activity_table['Threshold (min)'].isna(), '–',
                                    np.where(activity_table['Notebook (min)'] > activity_table['Limit (min)'], 'YES', 'no'))
activity_table.round(1)

## 6. Step by step vs the Process sheet
Every step in order, against its planned time in the workbook's Process sheet. *Empty Tank* and the acid fill steps have no planned time there.

In [ ]:
def normalise(text):
    return re.sub(r'\(.*?\)|[^a-z]', '', str(text).lower())

planned_seconds = {normalise(col): planned[col] for col in planned.index}
step_id = (records['Activity'] + records['Step']).ne((records['Activity'] + records['Step']).shift()).cumsum()
steps = records.groupby(step_id).agg(Activity=('Activity', 'first'), Step=('Step', 'first'),
                                     Start=('Date Time', 'first'), Records=('Date Time', 'size'),
                                     Counter=('Time In Seconds', 'first'))
steps['Actual (min)'] = ((steps['Counter'].shift(-1) - steps['Counter']) / 60  # until the next step starts
                         ).fillna(steps['Records'])  # last step: one record per minute
steps['Planned (min)'] = [planned_seconds.get(normalise(activity + step), np.nan) / 60
                          for activity, step in zip(steps['Activity'], steps['Step'])]
steps['Over Plan (min)'] = steps['Actual (min)'] - steps['Planned (min)']
steps['Start'] = steps['Start'].dt.strftime('%H:%M')
steps.drop(columns='Counter').reset_index(drop=True).round(1)

## 7. Did the wash work? (TACT during chemical circulation)
Time alone doesn't prove cleaning. For each chemical circulation: minutes with **temperature, conductivity and return flow** at set point, against the circulation time the Process sheet requires. Then: did the final rinse wash the chemical out?

In [ ]:
tact_rows = []
for activity, cond_sp, required in [('Caustic', 'Caustic SP', 'Caustic Circulation (Sec)'),
                                    ('Acid', 'Acid SP', 'Acid Circulation')]:
    circ = records[records['Activity'].eq(activity) & records['Step'].eq('Circulation')]
    temp_ok = circ['Return Temp'] >= circ['Temp SP']
    cond_ok = circ['Return Conducivity'] >= circ[cond_sp]
    flow_ok = circ['Return Flow'] >= circ['Min Return Flow SP']
    tact_rows.append({
        'Circulation': activity,
        'Minutes run': len(circ),
        'Required (min)': planned[required] / 60,
        'Median temp (°C)': circ['Return Temp'].median(),
        'Temp SP (°C)': circ['Temp SP'].median(),
        'Median conductivity (mS/cm)': circ['Return Conducivity'].median(),
        'Conductivity SP (mS/cm)': circ[cond_sp].median(),
        'Minutes temp OK': temp_ok.sum(),
        'Minutes conductivity OK': cond_ok.sum(),
        'Minutes return flow OK': flow_ok.sum(),
        'Minutes temp + conductivity OK': (temp_ok & cond_ok).sum(),
        'Minutes all three OK': (temp_ok & cond_ok & flow_ok).sum(),
    })
tact = pd.DataFrame(tact_rows).set_index('Circulation')
tact['Contact time met (temp + conductivity)'] = tact['Minutes temp + conductivity OK'] >= tact['Required (min)']

final_rinse = records[records['Activity'].eq('Final Rinse')]
rinse_end, rinse_sp = final_rinse['Return Conducivity'].iloc[-1], final_rinse['Final Rinse Cond. SP'].iloc[-1]
print(f"Final rinse ends at {rinse_end:.2f} mS/cm against a limit of {rinse_sp:.1f}: "
      + ('chemical rinsed out' if rinse_end <= rinse_sp else 'CHEMICAL MAY REMAIN'))
tact.round(1).T

## 8. Flow: what the alert uses vs circulation only
The flow alerts use the **median over the whole cycle**, including fill, drain and pump-off readings (about −90 L/hr). The circulation-only median shows whether a breach is real.

In [ ]:
circulation = records[records['Step'].eq('Circulation')]
flow_view = pd.DataFrame({
    flow: {
        'Threshold (L/hr)': cycle[f'{flow} Flow Threshold (L/hr)'],
        'Limit (L/hr)': cycle[f'{flow} Flow Threshold (L/hr)'] * (1 - FLOW_TOLERANCE),
        'Whole-cycle median (used by alert)': records[f'{flow} Flow'].median(),
        'Circulation median, positive readings': circulation.loc[circulation[f'{flow} Flow'] > 0, f'{flow} Flow'].median(),
        'Negative readings in cycle': records[f'{flow} Flow'].lt(0).sum(),
        'Circulation minutes below threshold': circulation[f'{flow} Flow'].lt(cycle[f'{flow} Flow Threshold (L/hr)']).sum(),
        'Circulation minutes': len(circulation),
    } for flow in FLOWS
})
breach_holds = flow_view.loc['Circulation median, positive readings'] < flow_view.loc['Limit (L/hr)']
flow_view = flow_view.round(0).astype(object)
flow_view.loc['Breach holds in circulation'] = breach_holds.map({True: 'yes', False: 'no'})
flips = records.loc[records['Flip'].astype(bool), ['Date Time', 'Activity', 'Step']]
print(f"Flip = 1 on {len(flips)} records: "
      + ', '.join(f"{row['Date Time']:%H:%M} {row['Activity']} {row['Step']}" for _, row in flips.iterrows()))
flow_view

## 9. The cycle over time
Minute-by-minute return temperature, conductivity and flow. Shaded bands mark the activities, and dashed lines are the set points the PLC logged. Rinse steps have no temperature set point.

In [ ]:
INK, MUTED, GRID, BAND = '#0b0b0b', '#52514e', '#e4e3df', '#efeeeb'
SUPPLY, RETURN = '#2a78d6', '#eb6834'  # validated categorical pair (slots 1 and 2)

conductivity_sp = np.select(
    [records['Activity'].eq('Caustic'), records['Activity'].eq('Acid'), records['Activity'].eq('Final Rinse')],
    [records['Caustic SP'], records['Acid SP'], records['Final Rinse Cond. SP']], np.nan)
panels = [
    ('Return temperature (°C)', [('Return temp', records['Return Temp'], INK)],
     ('Temp SP', records['Temp SP'].where(records['Temp SP'] > 1))),
    ('Return conductivity (mS/cm)', [('Return conductivity', records['Return Conducivity'], INK)],
     ('Chemical / final-rinse SP', conductivity_sp)),
    ('Flow (L/hr)', [('Supply flow', records['Supply Flow'], SUPPLY), ('Return flow', records['Return Flow'], RETURN)],
     ('Min return flow SP', records['Min Return Flow SP'])),
]
spans = by_activity['Date Time'].agg(['min', 'max'])
spans['max'] += pd.Timedelta(minutes=1)

fig, axes = plt.subplots(3, 1, figsize=(12, 8.5), sharex=True, constrained_layout=True)
for ax, (label, series, (sp_label, sp_values)) in zip(axes, panels):
    for i, (start, end) in enumerate(spans.itertuples(index=False)):
        if i % 2 == 0:
            ax.axvspan(start, end, color=BAND, lw=0)
    for name, values, color in series:
        ax.plot(records['Date Time'], values, color=color, lw=1.8, label=name)
    ax.step(records['Date Time'], sp_values, where='post', color=MUTED, lw=1.2, ls='--', label=sp_label)
    ax.set_ylabel(label, color=INK, fontsize=10)
    ax.grid(axis='y', color=GRID, lw=0.6)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.legend(frameon=False, fontsize=9, loc='upper right', ncol=3)
for activity, (start, end) in spans.iterrows():
    if (end - start) >= pd.Timedelta(minutes=6):
        axes[0].text(start + (end - start) / 2, 1.02, activity, transform=axes[0].get_xaxis_transform(),
                     ha='center', va='bottom', fontsize=9, color=MUTED)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
fig.suptitle(f"{WALK_OBJECT}, cycle {WALK_CYCLE} ({cycle['Cycle Start']:%d %b %Y})", x=0.01, ha='left',
             fontsize=12, color=INK)
plt.show()

**What this cycle shows** (cycle 14)
- **Caustic:** circulation starts at 00:00 but reaches 80 °C only at 00:12. Supply flow drops to ~0 in 17 of 48 minutes and return flow is below set point in 32 of the first 35. Circulation ran **48 min against 15 planned**.
- **Acid:** flows read ~0 for 17 minutes (01:20–01:39) and temperature stays ≤ 42 °C against 65. Acid reaches set point only from 01:38, giving **8 in-spec minutes against 7 required**.
- **Final rinse:** conductivity falls to 1.4 mS/cm (limit 2), so the chemical was rinsed out.
- **History (section 11):** 14–17 Jul brought four long caustic cycles in a row (53–87 min). This object's normal caustic time is 19–21 min.

## 10. Validation checks for this cycle
Rebuilds the cycle's numbers from its raw records, independently of the pipeline, and checks the data behind them. **CHECK** marks a point to discuss, not necessarily an error.

In [ ]:
cycle_checks = []
def cycle_check(name, passed, detail=''):
    cycle_checks.append({'Check': name, 'Result': 'PASS' if passed else 'CHECK', 'Detail': detail})

# Data
largest_gap = records['Date Time'].diff().dt.total_seconds().max()
cycle_check('Records are one minute apart with no gaps', largest_gap <= 90, f'largest gap {largest_gap:.0f} s')
cycle_check('No duplicate timestamps', not records['Date Time'].duplicated().any())
cycle_check('Elapsed counter only increases', records['Time In Seconds'].diff().dropna().ge(0).all())
cycle_check('Counter advances about 60 s per record',
            records['Time In Seconds'].diff().dropna().between(55, 65).mean() > 0.95,
            f"median step {records['Time In Seconds'].diff().median():.0f} s")
cycle_check('Activities run in the expected order',
            activity_order == [a for a in ACTIVITIES if a in activity_order], ' → '.join(activity_order))
cycle_check('Each activity is one continuous block',
            records['Activity'].ne(records['Activity'].shift()).sum() == records['Activity'].nunique())
cycle_check('Recipe is the same on every record', records['Recipe'].nunique() == 1, preview(records['Recipe'].unique()))
for sp in ['Min Return Flow SP', 'Caustic SP', 'Acid SP', 'Final Rinse Cond. SP']:
    cycle_check(f'{sp} is constant', records[sp].nunique() == 1, preview(records[sp].unique()))
for col, (low, high) in VALID_RANGES.items():
    outside = ~records[col].between(low, high)
    cycle_check(f'{col} within {low} to {high}', not outside.any(),
                f'{outside.sum()} readings outside, min {records[col].min():.1f}')

# Matching and thresholds
cycle_check('Matched on the same equipment code', object_key(WALK_OBJECT) == object_key(cycle['Reference Object']),
            f"{object_key(WALK_OBJECT)} vs {object_key(cycle['Reference Object'])}")
cycle_check('PLC Min Return Flow SP equals the workbook return flow',
            records['Min Return Flow SP'].iloc[0] == cycle['Return Flow Threshold (L/hr)'],
            f"{records['Min Return Flow SP'].iloc[0]:.0f} vs {cycle['Return Flow Threshold (L/hr)']:.0f}")
final_rinse_planned = planned[FINAL_RINSE_SECONDS].sum() / 60
cycle_check('Final Rinse threshold = sum of its 4 Process steps',
            np.isclose(final_rinse_planned, cycle['Final Rinse Threshold (min)']), f'{final_rinse_planned:.2f} min')

# Recalculation from raw records
for phase in ACTIVITIES:
    if phase in counters.index:
        recalculated = (counters.loc[phase, 'last'] - counters.loc[phase, 'first']) / 60
        cycle_check(f'{phase} time matches a recalculation', np.isclose(recalculated, cycle[phase]),
                    f'{recalculated:.2f} vs {cycle[phase]:.2f} min')
for flow in FLOWS:
    cycle_check(f'{flow} flow median matches a recalculation',
                np.isclose(records[f'{flow} Flow'].median(), cycle[f'{flow} Flow Actual (L/hr)']))
recalculated_alerts = (sum(cycle[phase] > cycle[f'{phase} Threshold (min)'] * (1 + DURATION_TOLERANCE)
                           for phase in DURATION_THRESHOLDS)
                       + sum(records[f'{flow} Flow'].median() < cycle[f'{flow} Flow Threshold (L/hr)'] * (1 - FLOW_TOLERANCE)
                             for flow in FLOWS))
cycle_check('Number of alerts matches a recalculation', recalculated_alerts == len(cycle_alerts),
            f'{recalculated_alerts} vs {len(cycle_alerts)}')

# Known issues, measured on this cycle
current_pre_rinse = summary.set_index('Object').loc[cycle['Reference Object'], 'Current-Total Pre rinse time(Min)']
cycle_check('Pre Rinse also breaches the timer in use in July (not just the proposed one)',
            cycle['Pre Rinse'] > current_pre_rinse * (1 + DURATION_TOLERANCE),
            f"{cycle['Pre Rinse']:.1f} min vs current {current_pre_rinse:.2f} min")
cycle_check('Acid breach holds when Mid Rinse is included',
            cycle[['Mid Rinse', 'Acid']].sum() > cycle['Acid Threshold (min)'] * (1 + DURATION_TOLERANCE),
            f"{cycle[['Mid Rinse', 'Acid']].sum():.1f} min vs {cycle['Acid Threshold (min)']:.2f}")
for flow in FLOWS:
    cycle_check(f'{flow} flow breach holds during circulation',
                not cycle[f'{flow} Flow Breach'] or bool(breach_holds[flow]),
                f"circulation median {flow_view.loc['Circulation median, positive readings', flow]:.0f} L/hr")
cycle_check('Final rinse ends below its conductivity limit', rinse_end <= rinse_sp, f'{rinse_end:.2f} mS/cm')

cycle_check_table = pd.DataFrame(cycle_checks)
print(cycle_check_table['Result'].value_counts().to_string())
with pd.option_context('display.max_rows', 100, 'display.max_colwidth', 80):
    show(cycle_check_table)

## 11. This object's other cycles
Is this cycle typical for the object? Every cycle of the same object, with the walkthrough cycle marked ▶.

In [ ]:
history = cycle_results[cycle_results['Object'].eq(WALK_OBJECT)][
    ['Cycle Number', 'Cycle Start', 'Recipes', *ACTIVITIES[:5], 'Return Flow Actual (L/hr)', 'Status', 'Alert Count']].copy()
history.insert(0, '', np.where(history['Cycle Number'].eq(WALK_CYCLE), '▶', ''))
history['Cycle Start'] = history['Cycle Start'].dt.strftime('%a %d %b %H:%M')
with pd.option_context('display.max_rows', 100):
    show(history.round(1).reset_index(drop=True))

## 12. Raw records
Every record in the cycle with its row in the source workbook, so any value can be checked against the original export.

In [ ]:
with pd.option_context('display.max_rows', 300):
    show(records[['Excel Row', 'Date Time', 'Activity', 'Step', 'Time In Seconds', 'Temp SP', 'Return Temp',
                  'Return Conducivity', 'Supply Flow', 'Return Flow', 'Min Return Flow SP', 'Flip']]
         .reset_index(drop=True))

## 13. Questions for the CIP owner
**About this cycle**
1. During caustic circulation, supply flow drops to ~0 in 17 of 48 minutes. Is that pulsed flow by design, or a pump or valve problem?
2. During acid circulation, flows read ~0 and the temperature stays at ~41 °C for 17 minutes (01:20–01:39). Was the circuit waiting, for example for the heater or a valve? Does the PLC count that time as circulation?
3. Does the PLC extend circulation until temperature and conductivity are in spec? That would explain caustic running 48 min against 15.
4. What happened on PMT-1 between 14 and 17 Jul, when four caustic cycles in a row ran 53–87 min?
5. What does `Flip = 1` mean? Here it is set at 00:12 (caustic) and 01:38 (acid).

**About the rules**

6. **Acid threshold:** `Acid Total Time` (17.7 min) includes Mid Rinse. Should we compare it with Mid Rinse + Acid (56 min here), or with acid circulation alone (7 min planned)?
7. **Pre-rinse:** for July data, should we use the proposed timer (7.9 min) or the timer in use then (8.8 min)?
8. **Flow:** the supply-flow alert fires on the whole-cycle median (12,156 L/hr), but the circulation median (13,920) is within limits. Should flow be judged on circulation only?
9. **Tolerances:** are the 10% duration and flow tolerances acceptable, or should they be set per object?